# Regional HAB forecasting with TabFM

Evaluate Google's pretrained tabular classifier on the same forecast target, 12 core
features, and historical years as Mohammed's boosting comparison. TabFM uses labeled
weeks as context and leaves the pretrained model weights fixed.

See the [TabFM model summary](../TabFM_Model_Development_Summary.md) for setup,
results, and the saved files.

## Load the regional dataset

Class 0 is below 5,000 cells/L. Class 1 is 5,000 to 99,999 cells/L. Class 2 is
at least 100,000 cells/L. The predictors describe the most recent available
observations through the feature week.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

here = Path.cwd()
root = next(
    (path for path in (here, *here.parents)
     if (path / "notebooks" / "hab_tabfm.py").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Open this notebook from the project checkout.")
sys.path.insert(0, str(root / "notebooks"))
data_dir = root / "data" / "final"

from hab_tabfm import (
    REPORT_WINDOWS, artifact_config, baseline_backtests, evaluate, forecast_folds,
    load_regional, load_tabfm, observed_forecast_weeks, save_tabfm_artifacts,
    tabfm_backtest, verify_tabfm_context,
)

raw, labeled, features = load_regional(data_dir)
observed = observed_forecast_weeks(labeled)
print(f"{len(raw)} regional weeks, {len(labeled)} labeled, {len(features)} core features")
display(labeled.groupby("split")["y_class"].value_counts().unstack(fill_value=0))
display(pd.Series(features, name="Core features").to_frame())

595 regional weeks, 486 labeled, 12 core features


y_class,0,1,2
split,,,
holdout,18,0,0
test,45,1,6
train,185,59,120
validate,38,5,9


,Core features
0,last_class
1,last_log_cells
2,last_class_prev
3,last_log_cells_chg1
4,weeks_since_obs
5,class_max_4wk
6,n_habsos_samples
7,weeks_since_bloom
8,pos_weeks_12wk
9,class_run_weeks


## Evaluation

The target is the next week's regional *K. brevis* severity. Evaluate only labeled
weeks. For each year from 2017 to 2024, use earlier years as training data and apply
a four-week gap before the first labeled target week. The 2020–2024 subset matches
notebook 07. For the 2025 test, train through 2024. The 2026 holdout is outside this run.

An **onset** is a target week at class 1 or 2 after the last observed class was 0.
A **severe miss** is class 2 called class 0. A **false alarm** is class 0 called
class 1 or 2. Onset recall is undefined in a window with zero onsets.
The cost scores add each false alarm to two or three times the severe-miss count.
Onset average precision ranks risk only on weeks whose last observed class was 0.

In [2]:
display(pd.DataFrame(
    [(year, len(train), len(test)) for year, train, test in forecast_folds(labeled)],
    columns=["Target year", "Training weeks", "Forecast weeks"],
))

,Target year,Training weeks,Forecast weeks
0,2017,99,52
1,2018,151,53
2,2019,204,52
3,2020,256,52
4,2021,308,27
5,2022,339,8
6,2023,346,17
7,2024,363,52
8,2025,416,52


## Build the TabFM forecasts

Use checkpoint `google/tabfm-1.0.0-pytorch` with 32 estimators and every available
training row as context. Add missing-value indicators for features with gaps in the
training fold. Compare with persistence, tuned LightGBM, and the CatBoost warning.
The first run downloads the public classification weights.

In [3]:
model = load_tabfm()
tabfm_forecasts = tabfm_backtest(labeled, features, model)
lightgbm_forecasts, catboost_forecasts = baseline_backtests(labeled, features)
forecasts = {
    "LightGBM": lightgbm_forecasts,
    "CatBoost": catboost_forecasts,
    "TabFM": tabfm_forecasts,
}

/Users/arniskc/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights from local directory


2017: 99 context weeks, 52 forecast weeks


2018: 151 context weeks, 53 forecast weeks


2019: 204 context weeks, 52 forecast weeks


2020: 256 context weeks, 52 forecast weeks


2021: 308 context weeks, 27 forecast weeks


2022: 339 context weeks, 8 forecast weeks


2023: 346 context weeks, 17 forecast weeks


2024: 363 context weeks, 52 forecast weeks


2025: 416 context weeks, 52 forecast weeks


## Alert threshold comparison

Use the class with highest probability for ordinary severity forecasts. For early
warning, use P(class 1) + P(class 2) and predict class 0 below the cutoff. Otherwise,
pick the more probable of classes 1 and 2. CatBoost uses the published cutoff of 0.35.

For each year from 2020 onward, count CatBoost's false alarms on earlier forecast
years. Use only those earlier class-0 weeks to choose a candidate risk cutoff that
aims for the same count. Apply the cutoff to the next year. The realized false-alarm
counts can differ. This compares alert policies on the historical data.

In [4]:
summary, weekly, cutoffs = evaluate(observed, forecasts)
for window in REPORT_WINDOWS:
    print(window)
    display(summary.loc[window].round(3))

Historical comparison (2020–2024)


,weeks,onsets caught,onsets,onset recall,severe misses,false alarms,cost 2:1,cost 3:1,accuracy,balanced accuracy,macro F1,onset AP,log loss
model,,,,,,,,,,,,,
Persistence,156,0,13,0.000,2,12,16,18,0.763,0.674,0.677,NaN,NaN
LightGBM,156,2,13,0.154,2,23,27,29,0.660,0.583,0.586,0.141,0.847
LightGBM (matched alert budget),156,6,13,0.462,0,53,53,53,0.506,0.547,0.518,0.141,0.847
CatBoost (0.35),156,6,13,0.462,0,33,33,33,0.660,0.659,0.636,0.205,0.654
TabFM,156,0,13,0.000,4,9,17,21,0.750,0.635,0.627,0.154,0.704
TabFM (matched alert budget),156,6,13,0.462,0,33,33,33,0.654,0.653,0.617,0.154,0.704


Extended backtest (2017–2024)


,weeks,onsets caught,onsets,onset recall,severe misses,false alarms,cost 2:1,cost 3:1,accuracy,balanced accuracy,macro F1,onset AP,log loss
model,,,,,,,,,,,,,
Persistence,313,0,25,0.00,5,25,35,40,0.786,0.704,0.704,NaN,NaN
LightGBM,313,6,25,0.24,6,38,50,56,0.722,0.654,0.651,0.230,0.704
CatBoost (0.35),313,13,25,0.52,1,63,65,66,0.684,0.686,0.654,0.298,0.620
TabFM,313,2,25,0.08,8,20,36,44,0.776,0.664,0.667,0.243,0.655


Final test (2025)


,weeks,onsets caught,onsets,onset recall,severe misses,false alarms,cost 2:1,cost 3:1,accuracy,balanced accuracy,macro F1,onset AP,log loss
model,,,,,,,,,,,,,
Persistence,52,0,0,NaN,0,1,1,1,0.962,0.659,0.637,NaN,NaN
LightGBM,52,0,0,NaN,0,3,3,3,0.904,0.589,0.600,NaN,0.300
LightGBM (matched alert budget),52,0,0,NaN,0,7,7,7,0.827,0.559,0.562,NaN,0.300
CatBoost (0.35),52,0,0,NaN,0,4,4,4,0.904,0.637,0.626,NaN,0.309
TabFM,52,0,0,NaN,0,1,1,1,0.962,0.659,0.637,NaN,0.207
TabFM (matched alert budget),52,0,0,NaN,0,3,3,3,0.923,0.644,0.630,NaN,0.207


## Compare which onsets were caught

In the historical comparison, inspect the overlap between CatBoost and the
TabFM alert policy. The class predictions are positive for both elevated and
severe risk.

In [5]:
history = weekly["target_year"].between(2020, 2024)
onset_weeks = weekly.loc[history & weekly["last_class"].eq(0) & weekly["y_class"].gt(0)]
cat = onset_weeks["CatBoost (0.35)"].gt(0)
tab = onset_weeks["TabFM (matched alert budget)"].gt(0)
print(f"{int((cat & tab).sum())} caught by both, "
      f"{int((cat & ~tab).sum())} by CatBoost only, "
      f"{int((~cat & tab).sum())} by TabFM only.")

4 caught by both, 2 by CatBoost only, 2 by TabFM only.


## Save and verify the TabFM artifacts

Save the weekly forecasts, evaluation tables, model configuration, and labeled
context through 2024. Reload the saved context, rebuild the classifier, and check
that its forecast for the first 2025 row matches the backtest. Notebook 09 uses
the saved TabFM probabilities.

In [6]:
output_dir = root / "models" / "tabfm"
config = artifact_config(data_dir, features)
save_tabfm_artifacts(
    output_dir, config, features, labeled, tabfm_forecasts, summary, weekly, cutoffs
)
sample = labeled.loc[labeled["target_year"] == 2025].head(1)
display(verify_tabfm_context(output_dir, model, sample, tabfm_forecasts).round(6))
print("Saved and verified models/tabfm/")

,p_class_0,p_class_1,p_class_2
target_week,,,
2025-01-06,0.032616,0.062704,0.90468


Saved and verified models/tabfm/
